# 02 — ETL Mossos d'Esquadra

Construye `fact_delitos_mossos` a partir de los fets penals de Mossos, haciendo lookup de IDs contra las dimensiones generadas en el notebook 01.

**Fuente:** `data/raw/mossos/mossos_fets_penals_raw.csv` (UTF-8, mensual, por ABP, 2011-2026)

**Output:** `data/clean/fact_delitos_mossos.csv`

**Columnas objetivo:** `id, id_tiempo, id_territorio, id_tipo_delito, coneguts, resolts, detencions`

**Reglas aplicadas:**
- Encoding UTF-8
- Filtrar `Any <= 2025` (hay datos parciales de 2026 — P9)
- `low_memory=False`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r'E:\informacion y documentos\Curso analisis de datos IT Academy\Proyecto final\criminalistica_cat')
RAW  = BASE / 'data' / 'raw'
CLEAN = BASE / 'data' / 'clean'

print('BASE:', BASE)
print('CLEAN existe:', CLEAN.exists())

---
## 1. Cargar Mossos raw

In [ ]:
mossos = pd.read_csv(
    RAW / 'mossos' / 'mossos_fets_penals_raw.csv',
    encoding='utf-8',
    low_memory=False
)
print('Shape inicial:', mossos.shape)
print('\nColumnas:', mossos.columns.tolist())
print('\nNulls por columna:')
print(mossos.isnull().sum())
mossos.head()

In [ ]:
print('Rango de años:', sorted(mossos['Any'].unique()))
print('\nFilas por año:')
print(mossos['Any'].value_counts().sort_index())

---
## 2. Filtrar Any <= 2025 (P9 — eliminar parciales de 2026)

In [ ]:
filas_antes = len(mossos)
mossos = mossos[mossos['Any'] <= 2025].copy()
print(f'Filas eliminadas (2026): {filas_antes - len(mossos)}')
print(f'Shape tras filtrar: {mossos.shape}')
print('Años restantes:', sorted(mossos['Any'].unique()))

---
## 3. Cargar dimensiones

In [ ]:
dim_tiempo = pd.read_csv(CLEAN / 'dim_tiempo.csv', encoding='utf-8')
dim_territorio = pd.read_csv(CLEAN / 'dim_territorio.csv', encoding='utf-8')
dim_tipo_delito = pd.read_csv(CLEAN / 'dim_tipo_delito.csv', encoding='utf-8')

print('dim_tiempo:', dim_tiempo.shape)
print('dim_territorio:', dim_territorio.shape)
print('dim_tipo_delito:', dim_tipo_delito.shape)

---
## 4. Lookup id_tiempo (por anyo + mes)

In [ ]:
mossos = mossos.merge(
    dim_tiempo[['id_tiempo', 'anyo', 'mes']],
    left_on=['Any', 'Mes'],
    right_on=['anyo', 'mes'],
    how='left'
)

n_sin_tiempo = mossos['id_tiempo'].isnull().sum()
print(f'Filas sin id_tiempo: {n_sin_tiempo}')
if n_sin_tiempo > 0:
    print('AVISO: combinaciones año/mes sin match:')
    print(mossos[mossos['id_tiempo'].isnull()][['Any', 'Mes']].drop_duplicates())

---
## 5. Lookup id_territorio (por ABP)

Se filtra `dim_territorio` a `fuente == 'mossos'` para que el ABP haga match con un único territorio.

In [ ]:
terr_mossos = dim_territorio[dim_territorio['fuente'] == 'mossos'][['id_territorio', 'abp']].copy()
print('Territorios mossos en dimensión:', len(terr_mossos))
print('ABPs únicos en datos:', mossos['Àrea Bàsica Policial (ABP)'].nunique())

mossos = mossos.merge(
    terr_mossos,
    left_on='Àrea Bàsica Policial (ABP)',
    right_on='abp',
    how='left'
)

n_sin_terr = mossos['id_territorio'].isnull().sum()
print(f'\nFilas sin id_territorio: {n_sin_terr}')
if n_sin_terr > 0:
    print('AVISO: ABPs sin match:')
    print(mossos[mossos['id_territorio'].isnull()]['Àrea Bàsica Policial (ABP)'].unique())

---
## 6. Lookup id_tipo_delito (por Títol Codi Penal + Tipus de fet)

Se hace match sobre la pareja `(titol_cp, descripcio)` filtrando a `fuente == 'mossos'`, porque un mismo `Tipus de fet` puede aparecer bajo distintos `Títol Codi Penal`.

In [ ]:
tipo_mossos = dim_tipo_delito[dim_tipo_delito['fuente'] == 'mossos'][['id_tipo_delito', 'titol_cp', 'descripcio']].copy()
print('Tipos mossos en dimensión:', len(tipo_mossos))

mossos = mossos.merge(
    tipo_mossos,
    left_on=['Títol Codi Penal', 'Tipus de fet'],
    right_on=['titol_cp', 'descripcio'],
    how='left'
)

n_sin_tipo = mossos['id_tipo_delito'].isnull().sum()
print(f'\nFilas sin id_tipo_delito: {n_sin_tipo}')
if n_sin_tipo > 0:
    print('AVISO: tipos sin match:')
    print(mossos[mossos['id_tipo_delito'].isnull()][['Títol Codi Penal', 'Tipus de fet']].drop_duplicates())

---
## 7. Construir tabla final

In [ ]:
fact = mossos.rename(columns={
    'Coneguts': 'coneguts',
    'Resolts': 'resolts',
    'Detencions': 'detencions'
})[['id_tiempo', 'id_territorio', 'id_tipo_delito', 'coneguts', 'resolts', 'detencions']].copy()

# Convertir IDs a int (tras verificar que no hay nulls)
assert fact[['id_tiempo', 'id_territorio', 'id_tipo_delito']].isnull().sum().sum() == 0, 'Hay lookups fallidos — revisar arriba'
fact['id_tiempo'] = fact['id_tiempo'].astype(int)
fact['id_territorio'] = fact['id_territorio'].astype(int)
fact['id_tipo_delito'] = fact['id_tipo_delito'].astype(int)

# PK incremental
fact.insert(0, 'id', range(1, len(fact) + 1))

print('Shape final:', fact.shape)
print('\nTipos de datos:')
print(fact.dtypes)
fact.head()

In [ ]:
print('Validaciones:')
print('  Nulls totales:', fact.isnull().sum().sum())
print('  Filas duplicadas (sin id):', fact.drop(columns='id').duplicated().sum())
print('  Suma coneguts:', fact['coneguts'].sum())
print('  Suma resolts:', fact['resolts'].sum())
print('  Suma detencions:', fact['detencions'].sum())
print('  Rango coneguts:', fact['coneguts'].min(), '-', fact['coneguts'].max())

---
## 8. Guardar

In [ ]:
ruta_salida = CLEAN / 'fact_delitos_mossos.csv'
fact.to_csv(ruta_salida, index=False, encoding='utf-8')
print(f'[OK] Guardado {ruta_salida.name}: {len(fact)} filas')
print('\nListo para continuar con 03_etl_gub_barcelona.ipynb')